# 🧪 Poker Arena — Notebook de QA (testes completos)

**Para a equipe de QA.** É **célula única**: dê **Run all** e leia o relatório no fim.

Roda **todos os tipos de teste da literatura de QA** sobre a solução: regressão (suite pytest + cobertura), análise estática, **teste de mesa** (passo a passo de uma mão), propriedade/invariantes, valor-limite, negativos, segurança, determinismo, integração de API e desempenho.

**Pré-requisito:** segredo `GH_TOKEN` (PAT do GitHub) nos Secrets do Colab.

In [ ]:
# ============================================================
# QA - Poker Arena | NOTEBOOK DE TESTES (celula unica - "Run all")
# Pre-requisito: segredo GH_TOKEN (PAT do GitHub) nos Secrets do Colab.
# ============================================================

# ---- 0. SETUP: clona o repo privado e instala ----
import os
from google.colab import userdata
gh = userdata.get('GH_TOKEN')
assert gh, 'Adicione o segredo GH_TOKEN (PAT do GitHub) nos Secrets do Colab.'
!rm -rf poker-arena
!git clone -q https://{gh}@github.com/FELIPEACASTRO/poker-arena.git
!pip install -q -e poker-arena/backend
!pip install -q pytest pytest-cov httpx ruff
print('Instalado. (NAO compartilhe a saida desta celula - contem o token)')
REPO = 'poker-arena/backend'

import random, time, subprocess
from fastapi.testclient import TestClient
from poker_arena.api.app import create_app
from poker_arena.api.dependencies import get_repository
from poker_arena.application import (BotSpec, InMemorySessionRepository,
    InvalidActionError, SessionConfig, UnknownBotLevel, build_session)
from poker_arena.engine.game import IllegalActionError

def box(titulo, linhas):
    w = max([len(titulo)] + [len(s) for s in linhas]) if linhas else len(titulo)
    print("\n  +" + "-" * (w + 2) + "+")
    print("  | " + titulo.ljust(w) + " |")
    print("  +" + "-" * (w + 2) + "+")
    for s in linhas:
        print("  | " + s.ljust(w) + " |")
    print("  +" + "-" * (w + 2) + "+")


_R: dict[str, list[int]] = {}


def check(categoria, cond, detalhe=""):
    p, f = _R.get(categoria, [0, 0])
    if cond:
        p += 1
    else:
        f += 1
        print(f"      [FALHA] {categoria}: {detalhe}")
    _R[categoria] = [p, f]


def _watch(n=6, stack=200, level="heuristic", rebuy=False, seed=0):
    return build_session(
        SessionConfig(
            bots=[BotSpec(f"B{i}", level) for i in range(n)],
            starting_stack=stack, mode="watch", rebuy=rebuy,
        ),
        seed=seed,
    )


def _play(n=5, stack=200, level="montecarlo", seed=0):
    return build_session(
        SessionConfig(
            bots=[BotSpec(f"B{i}", level) for i in range(n)],
            starting_stack=stack, mode="play", rebuy=True,
        ),
        seed=seed,
    )


# ============== 1. CONTEXTUALIZACAO PARA O TIME DE QA ==============
def contexto():
    box("PARA A EQUIPE DE QA — O QUE E A SOLUCAO", [
        "Poker Arena: Texas Hold'em No-Limit 6-max. Voce joga contra bots de",
        "inteligencia configuravel (Iniciante/Amador/Intermediario), ou ASSISTE",
        "bots jogando (cartas abertas). Arquitetura Limpa em 3 camadas:",
        "  engine (regras do poker)  ->  application (casos de uso)  ->  api (REST/WS).",
        "A dependencia aponta sempre pra dentro; a UI (React) consome a API.",
    ])
    box("O QUE VOCES VAO VALIDAR (tipos de teste da literatura)", [
        "Regressao .... a suite pytest inteira (unidade + integracao)",
        "Teste de mesa  passo a passo de UMA mao (desk check manual)",
        "Propriedade .. invariantes em centenas de maos (conservacao de fichas...)",
        "Valor-limite . heads-up, mesa cheia, all-in, recompra",
        "Negativo ..... entradas invalidas devem ser REJEITADAS",
        "Seguranca .... cartas do oponente NUNCA vazam no modo jogar",
        "Determinismo . mesma semente => mesmo resultado",
        "Integracao ... API ponta-a-ponta (TestClient)",
        "Desempenho ... vazao de maos por segundo",
    ])


# ============== 2. TESTE DE MESA (desk check) ==============
def teste_de_mesa():
    box("TESTE DE MESA — acompanhando UMA mao lance a lance", [
        "Voce (humano) sempre passa/paga; observe board, pote e stacks evoluirem.",
    ])
    s = _play(n=5, stack=200, seed=42)
    total0 = s.total_chips()  # 1 humano + 5 bots
    passo = 0
    while s.view().phase not in ("hand_over", "game_over") and passo < 80:
        v = s.view()
        legais = v.legal.actions if v.legal else []
        acao = "check" if "check" in legais else ("call" if "call" in legais else "fold")
        board = " ".join(v.board) if v.board else "(pre-flop)"
        print(f"   passo {passo:>2} | fase={v.phase:<10} pote={v.pot:>4} | board: {board}")
        # invariante visivel: total de fichas constante a cada passo
        check("Teste de mesa: conservacao por passo", s.total_chips() == total0,
              f"{s.total_chips()} != {total0}")
        s.apply_human_action(acao)
        passo += 1
    v = s.view()
    venc = ", ".join(v.seats[i].name for i in (v.winners or []))
    print(f"   >>> mao encerrada em '{v.phase}'. Vencedor(es): {venc}")
    check("Teste de mesa: mao chega a um fim valido", v.phase in ("hand_over", "game_over"))
    check("Teste de mesa: ha vencedor", bool(v.winners))


# ============== 3. PROPRIEDADE / INVARIANTES ==============
def prop(N=250):
    for t in range(N):
        s = _watch(n=6, stack=200, rebuy=False, seed=5000 + t)
        total0 = s.total_chips()
        guard = 0
        while s.view().phase == "bot_turn" and guard < 400:
            check("Propriedade: conservacao de fichas", s.total_chips() == total0,
                  f"trial {t}: {s.total_chips()} != {total0}")
            s.step()
            guard += 1
        v = s.view()
        cartas = list(v.board)
        for seat in v.seats:
            check("Propriedade: stack nunca negativo", seat.stack >= 0, f"trial {t}")
            if seat.cards:
                cartas += seat.cards
        check("Propriedade: board <= 5 cartas", len(v.board) <= 5, f"trial {t}")
        check("Propriedade: sem cartas duplicadas", len(cartas) == len(set(cartas)),
              f"trial {t}: {cartas}")


# ============== 4. VALOR-LIMITE / FRONTEIRA ==============
def fronteira():
    # heads-up (2 jogadores, o minimo)
    s = _watch(n=2, stack=200, rebuy=False, seed=11)
    g = 0
    while s.view().phase == "bot_turn" and g < 300:
        s.step(); g += 1
    check("Valor-limite: heads-up conserva fichas", s.total_chips() == 2 * 200)
    # mesa cheia + stacks minusculos forcando all-in e recompra (mesa segue cheia)
    s = _watch(n=6, stack=30, rebuy=True, seed=12)
    for _ in range(20):
        g = 0
        while s.view().phase == "bot_turn" and g < 300:
            s.step(); g += 1
        if s.view().phase == "hand_over":
            s.next_hand()
    check("Valor-limite: recompra mantem a mesa cheia (6)", len(s.view().seats) == 6)


# ============== 5. NEGATIVO (entradas invalidas rejeitadas) ==============
def negativos():
    # acao ilegal: passar quando ha aposta a pagar
    s = _play(n=5, seed=7)
    found = False
    for _ in range(40):
        v = s.view()
        if v.phase != "human_turn":
            break
        if v.legal and "check" not in v.legal.actions:
            try:
                s.apply_human_action("check")
                check("Negativo: acao ilegal rejeitada", False, "passar foi aceito indevidamente")
            except (InvalidActionError, IllegalActionError):
                check("Negativo: acao ilegal rejeitada", True)
            found = True
            break
        s.apply_human_action("check" if "check" in v.legal.actions else "call")
    if not found:
        check("Negativo: acao ilegal rejeitada", True)  # nao houve aposta; ok
    # nivel de bot inexistente
    try:
        build_session(SessionConfig(bots=[BotSpec("X", "supergto")]))
        check("Negativo: nivel desconhecido rejeitado", False)
    except UnknownBotLevel:
        check("Negativo: nivel desconhecido rejeitado", True)
    # step() no modo jogar (nao ha jogada de bot pendente)
    try:
        _play(seed=1).step()
        check("Negativo: step no modo jogar rejeitado", False)
    except InvalidActionError:
        check("Negativo: step no modo jogar rejeitado", True)


# ============== 6. SEGURANCA (cartas do oponente nao vazam) ==============
def seguranca():
    s = _play(n=5, seed=3)
    v = s.view()
    for seat in v.seats:
        if seat.kind == "human":
            check("Seguranca: humano ve as proprias cartas", seat.cards is not None)
        else:
            check("Seguranca: cartas do oponente ocultas (modo jogar)", seat.cards is None,
                  f"vazou seat {seat.seat}")


# ============== 7. DETERMINISMO (mesma semente => mesmo resultado) ==============
def determinismo():
    def joga(seed):
        s = _watch(n=6, stack=200, rebuy=False, seed=seed)
        g = 0
        while s.view().phase == "bot_turn" and g < 400:
            s.step(); g += 1
        v = s.view()
        return v.board, v.winners, [seat.stack for seat in v.seats]

    a = joga(999)
    b = joga(999)
    check("Determinismo: mesma semente => mesmo board/vencedor/stacks", a == b, f"{a} != {b}")


# ============== 8. INTEGRACAO DE API (TestClient) ==============
def integracao_api():
    app = create_app()
    repo = InMemorySessionRepository()
    app.dependency_overrides[get_repository] = lambda: repo
    client = TestClient(app)
    check("Integracao: GET /health", client.get("/health").status_code == 200)
    r = client.post("/tables", json={
        "bots": [{"name": f"B{i}", "level": "heuristic"} for i in range(5)],
        "starting_stack": 300, "seed": 7,
    })
    check("Integracao: POST /tables cria mesa (201)", r.status_code == 201)
    tid = r.json()["table_id"]
    check("Integracao: GET /tables/{id}", client.get(f"/tables/{tid}").status_code == 200)
    check("Integracao: GET mesa inexistente => 404", client.get("/tables/nope").status_code == 404)
    # joga ate um fim valido via HTTP
    ok = True
    for _ in range(60):
        st = client.get(f"/tables/{tid}").json()
        if st["phase"] == "human_turn":
            ok &= client.post(f"/tables/{tid}/actions", json={"type": "fold"}).status_code == 200
        elif st["phase"] == "hand_over":
            ok &= client.post(f"/tables/{tid}/next-hand").status_code == 200
        else:
            break
    check("Integracao: fluxo de jogo por HTTP sem erro", ok)
    # modo assistir via HTTP
    rw = client.post("/tables", json={
        "mode": "watch", "starting_stack": 200, "seed": 1,
        "bots": [{"name": f"W{i}", "level": "heuristic"} for i in range(6)],
    })
    check("Integracao: modo assistir cria mesa", rw.status_code == 201)
    twid = rw.json()["table_id"]
    g = 0
    while client.get(f"/tables/{twid}").json()["phase"] == "bot_turn" and g < 200:
        check("Integracao: POST /step (assistir)", client.post(f"/tables/{twid}/step").status_code == 200)
        g += 1


# ============== 9. DESEMPENHO (maos por segundo) ==============
def desempenho(M=200):
    t0 = time.perf_counter()
    maos = 0
    s = _watch(n=6, stack=500, rebuy=True, seed=77)
    while maos < M:
        g = 0
        while s.view().phase == "bot_turn" and g < 400:
            s.step(); g += 1
        if s.view().phase == "hand_over":
            s.next_hand()
            maos += 1
    dt = time.perf_counter() - t0
    print(f"   {maos} maos em {dt:.2f}s  =>  {maos/dt:,.0f} maos/seg")
    check("Desempenho: > 20 maos/seg", maos / dt > 20, f"{maos/dt:.0f}/s")


# ============== RELATORIO FINAL ==============
def relatorio():
    linhas = []
    tot_p = tot_f = 0
    for cat in sorted(_R):
        p, f = _R[cat]
        tot_p += p
        tot_f += f
        status = "PASS" if f == 0 else f"FALHOU x{f}"
        linhas.append(f"[{status:>8}] {cat}  ({p} ok)")
    box("RELATORIO DE QA - SIGN-OFF", linhas)
    veredito = "APROVADO - pode liberar" if tot_f == 0 else f"REPROVADO — {tot_f} falha(s)"
    box("VEREDITO FINAL", [f"asserts OK: {tot_p}", f"asserts FALHOS: {tot_f}", veredito])
    return tot_f == 0


# ============== 2. REGRESSAO (suite pytest + COBERTURA) ==============
def regressao():
    box('REGRESSAO - suite pytest inteira (unidade + integracao) + COBERTURA', [
        'Roda TODOS os testes automatizados do projeto e mede a cobertura de codigo.',
    ])
    r = subprocess.run(['python', '-m', 'pytest', REPO + '/tests', '-q',
                        '--cov=poker_arena', '--cov-report=term-missing:skip-covered'],
                       capture_output=True, text=True)
    print(r.stdout[-3500:])
    if r.stderr.strip():
        print(r.stderr[-600:])
    check('Regressao: suite pytest 100% verde', r.returncode == 0, 'ver saida acima')


def estatica():
    box('ANALISE ESTATICA - ruff (lint / clean code)', [
        'Verifica padroes de codigo, imports nao usados, estilo.',
    ])
    r = subprocess.run(['python', '-m', 'ruff', 'check', REPO],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'ruff: All checks passed!')
    check('Estatica: ruff lint limpo', r.returncode == 0, 'ver saida acima')


# ==================== EXECUCAO (ordem que o QA segue) ====================
random.seed(0)
contexto()
regressao()      # 2. suite automatizada inteira + cobertura
estatica()       # 2b. lint
teste_de_mesa()  # 3. desk check de UMA mao
box('TESTES DA LITERATURA DE QA (alem da suite automatizada)...', [
    'propriedade, valor-limite, negativo, seguranca, determinismo,',
    'integracao de API e desempenho.',
])
prop()
fronteira()
negativos()
seguranca()
determinismo()
integracao_api()
desempenho()
relatorio()
